# MMP-1 Ligand Molecular Dynamics

## Overview
This notebook runs short MD simulations of three candidate ligands (`ligand_2`, `ligand_8`, `ligand_9`) pre-placed in the binding pocket of MMP-1 (`1cgl_no_lig.pdb`).  
Each ligand is parametrized with **SMIRNOFF** (`openff-2.2.0`, via `openmmforcefields`) and the protein with **AMBER14**.

### Workflow per ligand
1. Load protein PDB + ligand PDB → merge with `Modeller`
2. Register SMIRNOFF template generator for the ligand residue
3. Add hydrogens, build system
4. Energy minimization
5. NVT equilibration
6. Production run → save `.dcd` trajectory
7. Visualize & compare

> **Note:** Simulations run in *vacuum* by default for speed.  
> Solvent cells are provided but commented out — see the `Solvent` section.

## 1. Imports

In [ ]:
# Core OpenMM
from openmm.app import (
    PDBFile, Modeller, ForceField, Simulation,
    DCDReporter, StateDataReporter, CheckpointReporter,
    NoCutoff, HBonds, PME
)
from openmm import LangevinIntegrator, MonteCarloBarostat
from openmm.unit import (
    kelvin, picosecond, picoseconds, nanometer,
    femtoseconds, bar
)

# SMIRNOFF small-molecule parametrization (no AmberTools needed on Windows)
from openmmforcefields.generators import SMIRNOFFTemplateGenerator
from openff.toolkit import Molecule

# Analysis & visualization
import MDAnalysis as mda
from md1 import visualize
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pathlib
import warnings
from sys import stdout

print("All imports OK.")

## 2. Global Configuration
Edit this cell to change which ligands to run, temperatures, or simulation length.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR    = pathlib.Path("data")
RESULTS_DIR = pathlib.Path("results")

PROTEIN_PDB = DATA_DIR / "1cgl_no_lig.pdb"   # MMP-1 structure (no ligand)

LIGAND_PDBS = [
    DATA_DIR / "ligand_2.pdb",
    DATA_DIR / "ligand_8.pdb",
    DATA_DIR / "ligand_9.pdb",
]

# ── Simulation Parameters ─────────────────────────────────────────────────
TEMPERATURE      = 310 * kelvin          # Physiological temperature
FRICTION         = 1 / picosecond
TIMESTEP         = 0.002 * picoseconds   # 2 fs — safe with HBonds constraints

EQUIL_TIME       = 10  * picoseconds     # NVT equilibration duration
PRODUCTION_TIME  = 100 * picoseconds     # Production run duration

TRAJ_RECORD_FREQ = 500                   # Save a frame every N steps
LOG_REPORT_FREQ  = 500

MINIMIZE_ITERS   = 200                   # Max energy minimization iterations

BOX = "vacuum"   # "vacuum" or "solvent" — change here, cells below adapt

# ── Derived ───────────────────────────────────────────────────────────────
EQUIL_STEPS      = int(EQUIL_TIME      / TIMESTEP)
PRODUCTION_STEPS = int(PRODUCTION_TIME / TIMESTEP)
TEMP_K           = int(TEMPERATURE.value_in_unit(kelvin))

print(f"Protein  : {PROTEIN_PDB}")
print(f"Ligands  : {[p.stem for p in LIGAND_PDBS]}")
print(f"Box      : {BOX}")
print(f"Temp     : {TEMPERATURE}")
print(f"Timestep : {TIMESTEP}")
print(f"Equil    : {EQUIL_STEPS} steps ({EQUIL_TIME})")
print(f"Prod     : {PRODUCTION_STEPS} steps ({PRODUCTION_TIME})")
print(f"Frames   : ~{PRODUCTION_STEPS // TRAJ_RECORD_FREQ} saved")

## 3. Core Simulation Function

The function below encapsulates the full MD pipeline for a single protein–ligand complex:
- Merge topologies with `Modeller`
- Register GAFF2 for the ligand residue
- Energy minimization → NVT equilibration → production
- Returns the `MDAnalysis.Universe` for trajectory analysis

In [ ]:
def run_mmp_ligand_sim(
    protein_pdb_path: pathlib.Path,
    ligand_pdb_path:  pathlib.Path,
    box:              str   = "vacuum",
    temperature             = 310 * kelvin,
    friction                = 1 / picosecond,
    timestep                = 0.002 * picoseconds,
    equil_steps:      int   = 5_000,
    production_steps: int   = 50_000,
    traj_record_freq: int   = 500,
    log_report_freq:  int   = 500,
    minimize_iters:   int   = 200,
    random_seed:      int   = 42,
    results_dir:      pathlib.Path = pathlib.Path("results"),
):
    """
    Run a protein-ligand MD simulation with AMBER14 + GAFF2.

    Parameters
    ----------
    protein_pdb_path : path to the prepared protein PDB (no ligand, no water)
    ligand_pdb_path  : path to the ligand PDB pre-placed in the binding pocket
    box              : "vacuum" or "solvent"
    Returns
    -------
    u : MDAnalysis.Universe loaded with the production trajectory
    out : dict of output file paths
    """
    ligand_name  = ligand_pdb_path.stem        # e.g. "ligand_2"
    protein_name = protein_pdb_path.stem       # e.g. "1cgl_no_lig"
    temp_K       = int(temperature.value_in_unit(kelvin))

    out_dir = results_dir / box / ligand_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*70}")
    print(f" Ligand: {ligand_name}  |  Protein: {protein_name}  |  Box: {box}  |  T: {temperature}")
    print(f"{'='*70}")

    # ── 1. Load structures ────────────────────────────────────────────────
    print("[1/6] Loading PDB files...")
    protein_pdb = PDBFile(str(protein_pdb_path))
    ligand_pdb  = PDBFile(str(ligand_pdb_path))

    # ── 2. GAFF2 parametrization for ligand ───────────────────────────────
    # OpenFF Toolkit reads the ligand PDB to build a proper Molecule object
    # (with bond orders) that GAFF2 needs. If your ligand PDB lacks CONECT
    # records, RDKit will infer bonds from geometry (usually works for small
    # drug-like molecules).
    print("[2/6] Parametrizing ligand with SMIRNOFF (openff-2.2.0)...")
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)
        openff_mol = Molecule.from_file(
            str(ligand_pdb_path),
            allow_undefined_stereo=True
        )
    smirnoff = SMIRNOFFTemplateGenerator(
        molecules=openff_mol,
        forcefield="openff-2.2.0"
    )

    # ── 3. Force field setup ─────────────────────────────────────────────
    print("[3/6] Building system...")
    if box == "solvent":
        forcefield = ForceField("amber14-all.xml", "amber14/tip3pfb.xml")
    else:
        forcefield = ForceField("amber14-all.xml")
    forcefield.registerTemplateGenerator(smirnoff.generator)

    # Merge protein + ligand into one Modeller topology
    modeller = Modeller(protein_pdb.topology, protein_pdb.positions)
    modeller.add(ligand_pdb.topology, ligand_pdb.positions)
    modeller.deleteWater()                 # remove any crystal waters in the PDB
    modeller.addHydrogens(forcefield)      # add missing H using AMBER protonation

    print(f"   Complex atoms (protein + ligand + H): {modeller.topology.getNumAtoms()}")

    if box == "solvent":
        modeller.addSolvent(forcefield, padding=1.0 * nanometer)
        print(f"   Solvated atoms: {modeller.topology.getNumAtoms()}")
        system = forcefield.createSystem(
            modeller.topology,
            nonbondedMethod=PME,
            nonbondedCutoff=1.0 * nanometer,
            constraints=HBonds,
            rigidWater=True,
        )
    else:
        system = forcefield.createSystem(
            modeller.topology,
            nonbondedMethod=NoCutoff,
            constraints=HBonds,
        )

    # ── 4. Integrator & Simulation object ────────────────────────────────
    integrator = LangevinIntegrator(temperature, friction, timestep)
    integrator.setRandomNumberSeed(random_seed)

    simulation = Simulation(modeller.topology, system, integrator)
    simulation.context.setPositions(modeller.positions)
    simulation.context.setVelocitiesToTemperature(temperature)

    # Save topology PDB (needed to load trajectory later)
    topology_pdb_path = out_dir / f"{ligand_name}_{box}_topology.pdb"
    positions = simulation.context.getState(getPositions=True).getPositions()
    with open(topology_pdb_path, "w") as f:
        PDBFile.writeFile(simulation.topology, positions, f)

    # ── 5. Energy minimization ───────────────────────────────────────────
    print("[4/6] Energy minimization...")
    e_before = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    print(f"   E_before = {e_before:.2f}")
    simulation.minimizeEnergy(maxIterations=minimize_iters)
    e_after = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    print(f"   E_after  = {e_after:.2f}")

    # ── 6. NVT equilibration (no trajectory saved) ───────────────────────
    print(f"[5/6] NVT equilibration ({equil_steps} steps)...")
    simulation.context.setVelocitiesToTemperature(temperature)
    simulation.step(equil_steps)
    print("   Equilibration complete.")

    # ── 7. Production run ────────────────────────────────────────────────
    print(f"[6/6] Production MD ({production_steps} steps)...")
    traj_path       = out_dir / f"{ligand_name}_{temp_K}K_traj.dcd"
    log_path        = out_dir / f"{ligand_name}_{temp_K}K_log.csv"
    checkpoint_path = out_dir / f"{ligand_name}_{temp_K}K_checkpoint.chk"

    simulation.reporters = []
    simulation.reporters.append(
        DCDReporter(str(traj_path), traj_record_freq, enforcePeriodicBox=(box == "solvent"))
    )
    simulation.reporters.append(
        StateDataReporter(
            stdout, production_steps // 5,
            step=True, potentialEnergy=True, temperature=True, speed=True
        )
    )
    simulation.reporters.append(
        StateDataReporter(
            str(log_path), log_report_freq,
            step=True, potentialEnergy=True, kineticEnergy=True,
            totalEnergy=True, temperature=True, speed=True, separator=","
        )
    )
    simulation.reporters.append(
        CheckpointReporter(str(checkpoint_path), traj_record_freq)
    )

    print(f"   Platform: {simulation.context.getPlatform().getName()}")
    simulation.step(production_steps)

    for r in simulation.reporters:
        if hasattr(r, "close"):
            r.close()

    # Save final structure
    final_pdb_path = out_dir / f"{ligand_name}_{temp_K}K_final.pdb"
    state = simulation.context.getState(getPositions=True)
    with open(final_pdb_path, "w") as f:
        PDBFile.writeFile(simulation.topology, state.getPositions(), f)

    print(f"   Trajectory : {traj_path}")
    print(f"   Log        : {log_path}")
    print(f"   Final PDB  : {final_pdb_path}")

    # Load trajectory for analysis/visualization
    u = mda.Universe(str(topology_pdb_path), str(traj_path))

    out_paths = {
        "topology": topology_pdb_path,
        "trajectory": traj_path,
        "log": log_path,
        "final_pdb": final_pdb_path,
    }
    return u, out_paths

print("run_mmp_ligand_sim() defined.")

## 4. Run Simulations for All Three Ligands

This cell iterates over the three candidate ligands and calls `run_mmp_ligand_sim()` for each.  
Results (trajectories, logs, checkpoints) are written to `results/<box>/<ligand_name>/`.

In [ ]:
universes  = {}   # ligand_name -> MDAnalysis.Universe
out_paths  = {}   # ligand_name -> dict of file paths

for ligand_path in LIGAND_PDBS:
    u, paths = run_mmp_ligand_sim(
        protein_pdb_path  = PROTEIN_PDB,
        ligand_pdb_path   = ligand_path,
        box               = BOX,
        temperature       = TEMPERATURE,
        friction          = FRICTION,
        timestep          = TIMESTEP,
        equil_steps       = EQUIL_STEPS,
        production_steps  = PRODUCTION_STEPS,
        traj_record_freq  = TRAJ_RECORD_FREQ,
        log_report_freq   = LOG_REPORT_FREQ,
        minimize_iters    = MINIMIZE_ITERS,
        results_dir       = RESULTS_DIR,
    )
    universes[ligand_path.stem] = u
    out_paths[ligand_path.stem] = paths

print("\nAll simulations complete.")
print("Ligands simulated:", list(universes.keys()))

## 5. Trajectory Visualization

Use the cells below to inspect each ligand's trajectory interactively.  
The protein backbone is shown as a cartoon colored by chain; the ligand residue is shown as sticks.

### 5a. Ligand 2

In [ ]:
view = visualize(universes["ligand_2"], step=1, max_frames=150)
view

### 5b. Ligand 8

In [ ]:
view = visualize(universes["ligand_8"], step=1, max_frames=150)
view

### 5c. Ligand 9

In [ ]:
view = visualize(universes["ligand_9"], step=1, max_frames=150)
view

## 6. Energy Analysis

Plot potential energy and temperature over time for each ligand to verify the simulation stayed stable.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=False)
ligand_names = [p.stem for p in LIGAND_PDBS]

for col, name in enumerate(ligand_names):
    log_file = out_paths[name]["log"]
    df = pd.read_csv(log_file)

    # Rename columns — StateDataReporter uses verbose headers
    df.columns = [c.strip().replace('"', '') for c in df.columns]

    # Identify the step and energy columns regardless of exact header text
    step_col  = [c for c in df.columns if "Step" in c][0]
    pe_col    = [c for c in df.columns if "Potential" in c][0]
    temp_col  = [c for c in df.columns if "Temperature" in c][0]

    time_ns = df[step_col] * TIMESTEP.value_in_unit(picoseconds) / 1000  # ns

    ax_e = axes[0, col]
    ax_t = axes[1, col]

    ax_e.plot(time_ns, df[pe_col], lw=0.8, color="steelblue")
    ax_e.set_title(f"{name}\nPotential Energy", fontsize=10)
    ax_e.set_xlabel("Time (ns)")
    ax_e.set_ylabel("E_pot (kJ/mol)")

    ax_t.plot(time_ns, df[temp_col], lw=0.8, color="tomato")
    ax_t.axhline(TEMP_K, ls="--", color="k", lw=0.7, label=f"Target {TEMP_K} K")
    ax_t.set_title(f"{name}\nTemperature", fontsize=10)
    ax_t.set_xlabel("Time (ns)")
    ax_t.set_ylabel("T (K)")
    ax_t.legend(fontsize=8)

plt.suptitle(f"MMP-1 Ligand MD — {BOX} — {TEMP_K} K", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"energy_temperature_{BOX}_{TEMP_K}K.png", dpi=150)
plt.show()
print("Plot saved.")

## 7. Ligand RMSD from Starting Position

RMSD of ligand heavy atoms relative to frame 0 measures how much each ligand drifts from its initial docked pose.  
A stable binder should show low RMSD; a ligand that leaves the pocket will show a large, sustained increase.

In [ ]:
from MDAnalysis.analysis import rms

fig, ax = plt.subplots(figsize=(8, 4))

colors = ["steelblue", "darkorange", "forestgreen"]

for (name, u), color in zip(universes.items(), colors):
    # Select ligand heavy atoms (non-protein, non-H)
    # Adjust the resname selector to match your ligand residue name in the PDB
    lig_sel = u.select_atoms("not protein and not resname HOH WAT SOL and not name H*")

    if len(lig_sel) == 0:
        print(f"  WARNING: No ligand atoms found for {name}. Check resname in topology PDB.")
        continue

    print(f"  {name}: {len(lig_sel)} heavy atoms selected for RMSD")

    rmsd_analysis = rms.RMSD(
        lig_sel,
        lig_sel,
        ref_frame=0
    )
    rmsd_analysis.run()

    time_ps = rmsd_analysis.results.rmsd[:, 1]   # time in ps
    rmsd_A  = rmsd_analysis.results.rmsd[:, 2]   # RMSD in Angstrom

    ax.plot(time_ps, rmsd_A, label=name, color=color, lw=1.2)

ax.set_xlabel("Time (ps)")
ax.set_ylabel("Ligand RMSD (Å)")
ax.set_title(f"Ligand Heavy-Atom RMSD vs Frame 0 — {BOX} {TEMP_K} K")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"ligand_rmsd_{BOX}_{TEMP_K}K.png", dpi=150)
plt.show()
print("RMSD plot saved.")

## 8. Final Structure Visualization

Show the last frame of each simulation to visually inspect the final pose.

In [ ]:
for name in ligand_names:
    topology_path = out_paths[name]["topology"]
    final_path    = out_paths[name]["final_pdb"]
    u_final = mda.Universe(str(topology_path), str(final_path))
    print(f"Final frame — {name} at {TEMP_K} K")
    view = visualize(u_final, step=1, max_frames=1)
    display(view)

## 9. (Optional) Solvent Simulation

To run in explicit TIP3P-FB water, change `BOX = "solvent"` in **Section 2** and rerun from there.  
This roughly 10× increases atom count and run time; it may require a GPU or significant RAM.

The `run_mmp_ligand_sim()` function handles both environments transparently:  
- Vacuum → `NoCutoff`, no PME, no barostat  
- Solvent → `PME`, 1 nm cutoff, `tip3pfb.xml`, water box padded 1 nm around the complex

For NPT equilibration in solvent, add a `MonteCarloBarostat` before production:

```python
# After NVT equilibration, add barostat for NPT:
barostat = MonteCarloBarostat(1.0 * bar, TEMPERATURE, 25)
system.addForce(barostat)
simulation.context.reinitialize(preserveState=True)
simulation.step(NPT_STEPS)
```

## 10. Troubleshooting

| Problem | Likely cause | Fix |
|---|---|---|
| `ValueError: No template found for residue LIG` | SMIRNOFF couldn't match ligand atom types | Ensure ligand PDB has CONECT records; try `allow_undefined_stereo=True` |
| `OpenFF Toolkit` can't parse PDB | Missing bond information | Convert ligand to SDF/MOL2 with RDKit and use `Molecule.from_file('ligand.sdf')` |
| Huge positive energies after minimization | Steric clashes between protein and ligand | Check docking pose quality; consider re-docking |
| `NaN` in energy log | Simulation exploded (too large forces) | Reduce timestep to `0.001 ps`, increase minimization iters, or fix clashes |
| RMSD selector returns 0 atoms | Ligand residue name mismatch | Open topology PDB in a text editor; find the ligand residue name and update `rms.RMSD` selector |
